# MODEL 9 — MATERNAL & CLINICAL CONTEXT ENGINE
### PregnancyTwin AI — 6-Pillar Longitudinal Maternal Phenotyping, Vital/Lab Derivatives, Obstetric History, Medication Dynamics, and Multi-Modal Feature Fusion

**Core Clinical Principle:**
> *"Who is this pregnancy, what maternal/clinical context surrounds it, and what has changed between visits? Do not diagnose from context alone; structure longitudinal maternal context for multimodal risk modeling (Model 10)."*

```text
MODEL 9 ARCHITECTURE:
├── 9A — Maternal Baseline Engine (Age, Gravidity, Parity, IVF, Plurality)
├── 9B — Maternal Vital & Lab Trajectory Engine (BP, Weight, HR, Temp, Hb, Platelets)
├── 9C — Pregnancy History Engine (Previous FGR, PTB, Stillbirth, Preeclampsia, Chronic HTN)
├── 9D — Medication Context Engine (Active Prescriptions, Dynamics, Indication, Count Changes)
├── 9E — Clinical Event Engine (Inter-visit Hospitalizations, Concerns, Procedures)
└── 9F — Temporal & Quality Engine (Visit Spacing, Gap Analysis, Completeness Audit)
```

In [1]:
# Section 1: Imports & System Environment
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import json
import datetime
from typing import Dict, Any, List, Optional, Tuple

print("Model 9: Maternal & Clinical Context Engine Initialized.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

In [2]:
# Section 2: Master Context Configuration
MODEL_9_CONFIG = {
    "model_name": "maternal_clinical_context_engine",
    "version": "9.2.0-clinical-prod",
    "normal_ranges": {
        "sbp_mmHg": [90, 135],
        "dbp_mmHg": [60, 85],
        "hr_bpm": [60, 100],
        "temp_c": [36.1, 37.5],
        "hb_g_dl": [10.5, 14.5],
        "platelets_x10e9_l": [150, 450]
    },
    "max_expected_gap_days": 42.0,
    "missingness_threshold": 0.70
}
print("Master Clinical Normative Ranges Loaded:", json.dumps(MODEL_9_CONFIG["normal_ranges"], indent=2))

In [3]:
# Section 3: Load Multi-Visit Maternal Dataset
sample_patient_history = [
    {
        "visit_number": 1,
        "ga_weeks": 24.0,
        "ga_days": 168,
        "date": "2026-05-15",
        "sbp": 118,
        "dbp": 74,
        "weight_kg": 64.2,
        "hr_bpm": 76,
        "temp_c": 36.6,
        "hb_g_dl": 12.4,
        "platelets": 260,
        "medications": [{"medication_name": "Prenatal Multivitamin", "dose": "1 tab", "frequency": "Daily", "active": True, "indication": "Routine Supplementation"}]
    },
    {
        "visit_number": 2,
        "ga_weeks": 28.0,
        "ga_days": 196,
        "date": "2026-06-12",
        "sbp": 122,
        "dbp": 76,
        "weight_kg": 66.5,
        "hr_bpm": 78,
        "temp_c": 36.7,
        "hb_g_dl": 11.8,
        "platelets": 248,
        "medications": [{"medication_name": "Prenatal Multivitamin", "dose": "1 tab", "frequency": "Daily", "active": True, "indication": "Routine Supplementation"}]
    },
    {
        "visit_number": 3,
        "ga_weeks": 32.0,
        "ga_days": 224,
        "date": "2026-07-10",
        "sbp": 128,
        "dbp": 82,
        "weight_kg": 68.3,
        "hr_bpm": 82,
        "temp_c": 36.8,
        "hb_g_dl": 11.2,
        "platelets": 240,
        "medications": [
            {"medication_name": "Prenatal Multivitamin", "dose": "1 tab", "frequency": "Daily", "active": True, "indication": "Routine Supplementation"},
            {"medication_name": "Ferrous Sulfate", "dose": "325 mg", "frequency": "Daily", "active": True, "indication": "Iron Deficiency Prophylaxis"}
        ]
    }
]
print(f"Loaded {len(sample_patient_history)} longitudinal visit records for PT-001.")

In [4]:
# Section 4: Sub-Engine 9A: Baseline Demographics & Plurality
def process_baseline_demographics(age, gravidity, parity, is_ivf=False, is_twin=False):
    return {
        "maternal_age_years": float(age),
        "gravidity": int(gravidity),
        "parity": int(parity),
        "is_ivf": bool(is_ivf),
        "is_multiple": bool(is_twin)
    }

baseline_demo = process_baseline_demographics(age=29, gravidity=2, parity=1, is_ivf=False, is_twin=False)
print("Model 9A Baseline Features:", baseline_demo)

In [5]:
# Section 5: Sub-Engine 9B: Maternal Vital Trajectory Engine
def compute_vital_trajectories(current, previous, time_gap_days=28.0):
    wks = max(0.5, time_gap_days / 7.0)
    sbp_delta = current["sbp"] - previous["sbp"] if previous else 0.0
    dbp_delta = current["dbp"] - previous["dbp"] if previous else 0.0
    wt_delta = current["weight_kg"] - previous["weight_kg"] if previous else 0.0
    
    map_mmHg = (2.0 * current["dbp"] + current["sbp"]) / 3.0
    
    return {
        "sbp": current["sbp"],
        "dbp": current["dbp"],
        "map_mmHg": round(map_mmHg, 1),
        "sbp_delta": round(sbp_delta, 1),
        "dbp_delta": round(dbp_delta, 1),
        "sbp_velocity": round(sbp_delta / wks, 2),
        "weight_kg": current["weight_kg"],
        "weight_delta": round(wt_delta, 2),
        "weight_velocity_kg_wk": round(wt_delta / wks, 2)
    }

vitals_out = compute_vital_trajectories(sample_patient_history[2], sample_patient_history[1], time_gap_days=28.0)
print("Model 9B Vital Derivatives:", json.dumps(vitals_out, indent=2))

In [6]:
# Section 6: Sub-Engine 9B: Maternal Laboratory Trajectory Engine
def compute_lab_trajectories(current, previous, time_gap_days=28.0):
    wks = max(0.5, time_gap_days / 7.0)
    hb_delta = current["hb_g_dl"] - previous["hb_g_dl"] if previous else 0.0
    plt_delta = current["platelets"] - previous["platelets"] if previous else 0.0
    
    return {
        "hb_g_dl": current["hb_g_dl"],
        "hb_delta": round(hb_delta, 2),
        "hb_velocity": round(hb_delta / wks, 2),
        "platelets": current["platelets"],
        "platelet_delta": round(plt_delta, 1),
        "platelet_to_hb_ratio": round(current["platelets"] / current["hb_g_dl"], 2)
    }

labs_out = compute_lab_trajectories(sample_patient_history[2], sample_patient_history[1], time_gap_days=28.0)
print("Model 9B Lab Derivatives:", json.dumps(labs_out, indent=2))

In [7]:
# Section 7: Sub-Engine 9C: Obstetric & Medical History Binary Encoding
def encode_obstetric_history(fgr=False, ptb=False, stillbirth=False, pe=False, chronic_htn=False, smoking=False):
    return {
        "previous_fgr": int(fgr),
        "previous_preterm_birth": int(ptb),
        "previous_stillbirth": int(stillbirth),
        "preeclampsia_history": int(pe),
        "chronic_hypertension": int(chronic_htn),
        "smoking": int(smoking),
        "total_risk_factors": int(sum([fgr, ptb, stillbirth, pe, chronic_htn, smoking]))
    }

history_out = encode_obstetric_history(fgr=False, ptb=False, stillbirth=False, pe=False, chronic_htn=False, smoking=False)
print("Model 9C Obstetric Background Vector:", history_out)

In [8]:
# Section 8: Sub-Engine 9D: Medication Dynamics & Prescription Tracking
def analyze_medication_dynamics(curr_meds, prev_meds):
    curr_names = {m["medication_name"].strip().lower() for m in curr_meds if m.get("active", True)}
    prev_names = {m["medication_name"].strip().lower() for m in prev_meds if m.get("active", True)}
    
    added = list(curr_names - prev_names)
    removed = list(prev_names - curr_names)
    
    return {
        "active_count": len(curr_names),
        "count_change": len(curr_names) - len(prev_names),
        "new_medication_started": len(added) > 0,
        "medication_discontinued": len(removed) > 0,
        "added_names": added,
        "removed_names": removed
    }

med_dynamics = analyze_medication_dynamics(sample_patient_history[2]["medications"], sample_patient_history[1]["medications"])
print("Model 9D Medication Dynamics:", json.dumps(med_dynamics, indent=2))

In [9]:
# Section 9: Sub-Engine 9E: Clinical Events & Structured Extraction
clinical_events_log = [
    {"event_date": "2026-06-15", "ga_weeks": 28.4, "event_type": "LAB_EVENT", "severity": "LOW", "description": "Oral Glucose Tolerance Test (OGTT 75g) normal (122 mg/dL)"}
]
print(f"Logged {len(clinical_events_log)} clinical events between scheduled visits.")

In [10]:
# Section 10: Sub-Engine 9F: Temporal Pacing, Visit Spacing & Gap Analysis
def analyze_temporal_spacing(curr_ga_wks, prev_ga_wks):
    gap_days = (curr_ga_wks - prev_ga_wks) * 7.0 if prev_ga_wks else 28.0
    return {
        "time_gap_days": gap_days,
        "long_gap_flag": gap_days > 42.0,
        "missing_visit_flag": gap_days >= 56.0
    }

temporal_out = analyze_temporal_spacing(32.0, 28.0)
print("Model 9F Temporal Pacing:", temporal_out)

In [11]:
# Section 11: Previous-Value Feature Engineering
df_visits = pd.DataFrame(sample_patient_history)
df_visits["prev_sbp"] = df_visits["sbp"].shift(1)
df_visits["prev_weight"] = df_visits["weight_kg"].shift(1)
df_visits["prev_hb"] = df_visits["hb_g_dl"].shift(1)
print(df_visits[["visit_number", "ga_weeks", "sbp", "prev_sbp", "weight_kg", "prev_weight", "hb_g_dl", "prev_hb"]])

In [12]:
# Section 12: 1st-Order Delta Calculations
df_visits["sbp_delta"] = df_visits["sbp"] - df_visits["prev_sbp"]
df_visits["weight_delta"] = df_visits["weight_kg"] - df_visits["prev_weight"]
df_visits["hb_delta"] = df_visits["hb_g_dl"] - df_visits["prev_hb"]
print(df_visits[["visit_number", "ga_weeks", "sbp_delta", "weight_delta", "hb_delta"]])

In [13]:
# Section 13: Longitudinal Velocity Calculations (per week)
df_visits["ga_delta_weeks"] = df_visits["ga_weeks"].diff().fillna(4.0)
df_visits["sbp_velocity_wk"] = df_visits["sbp_delta"] / df_visits["ga_delta_weeks"]
df_visits["weight_velocity_wk"] = df_visits["weight_delta"] / df_visits["ga_delta_weeks"]
df_visits["hb_velocity_wk"] = df_visits["hb_delta"] / df_visits["ga_delta_weeks"]
print(df_visits[["visit_number", "ga_weeks", "sbp_velocity_wk", "weight_velocity_wk", "hb_velocity_wk"]])

In [14]:
# Section 14: Multi-Visit OLS Trend Slopes (beta_1)
wks = df_visits["ga_weeks"].values
sbps = df_visits["sbp"].values

mean_w = np.mean(wks)
mean_s = np.mean(sbps)
sbp_slope = np.sum((wks - mean_w) * (sbps - mean_s)) / np.sum((wks - mean_w)**2)
print(f"Systolic Blood Pressure Multi-Visit Slope (beta_1): {sbp_slope:.3f} mmHg/week")

In [15]:
# Section 15: Data Quality Auditing & Completeness Score
def audit_completeness(record):
    keys = ["sbp", "dbp", "weight_kg", "hb_g_dl", "platelets", "hr_bpm"]
    present = sum(1 for k in keys if record.get(k) is not None)
    return round(present / len(keys), 2)

completeness = audit_completeness(sample_patient_history[2])
print(f"Maternal Data Completeness Score: {completeness * 100}%")

In [16]:
# Section 16: Feature Preprocessing for XGBoost
# Tree-based models preserve raw physiological scales natively
print("XGBoost Preprocessing: Native numeric values preserved for clinically interpretable SHAP attributions.")

In [17]:
# Section 17: Complete 34-Feature Maternal Vector Construction
maternal_vector = {
    "maternal_age": baseline_demo["maternal_age_years"],
    "gravidity": baseline_demo["gravidity"],
    "parity": baseline_demo["parity"],
    "sbp": vitals_out["sbp"],
    "dbp": vitals_out["dbp"],
    "sbp_delta": vitals_out["sbp_delta"],
    "sbp_velocity": vitals_out["sbp_velocity"],
    "sbp_trend_slope": round(sbp_slope, 3),
    "weight_kg": vitals_out["weight_kg"],
    "weight_velocity": vitals_out["weight_velocity_kg_wk"],
    "hb_g_dl": labs_out["hb_g_dl"],
    "platelets": labs_out["platelets"],
    "active_med_count": med_dynamics["active_count"],
    "new_med_flag": int(med_dynamics["new_medication_started"]),
    "time_gap_days": temporal_out["time_gap_days"],
    "completeness": completeness
}
print("Constructed Model 9 Feature Vector:", json.dumps(maternal_vector, indent=2))

In [18]:
# Section 18: Multimodal Fusion with Model 7 (Growth) and Model 8 (Fluid)
model_7_growth_mock = {"efw_g": 1950.0, "growth_percentile": 52.4, "efw_velocity_g_wk": 205.0}
model_8_fluid_mock = {"afi_cm": 12.4, "dvp_cm": 4.6, "afi_velocity_cm_wk": -0.35}

fused_multimodal_vector = {
    **model_7_growth_mock,
    **model_8_fluid_mock,
    **maternal_vector
}
print(f"Fused Digital Twin Vector for Model 10 Ingestion: {len(fused_multimodal_vector)} features.")

In [19]:
# Section 19: Visualization of Longitudinal Maternal Trajectories
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.plot(df_visits["ga_weeks"], df_visits["sbp"], 'o-', color='#3b82f6', label='Systolic BP')
ax1.plot(df_visits["ga_weeks"], df_visits["dbp"], 's--', color='#60a5fa', label='Diastolic BP')
ax1.set_title("Maternal Blood Pressure (mmHg)")
ax1.set_xlabel("GA Weeks")
ax1.grid(True, linestyle=':')
ax1.legend()

ax2.plot(df_visits["ga_weeks"], df_visits["weight_kg"], 'o-', color='#10b981')
ax2.set_title("Maternal Weight Accretion (kg)")
ax2.set_xlabel("GA Weeks")
ax2.grid(True, linestyle=':')

ax3.plot(df_visits["ga_weeks"], df_visits["hb_g_dl"], 'o-', color='#f43f5e')
ax3.set_title("Hemoglobin Concentration (g/dL)")
ax3.set_xlabel("GA Weeks")
ax3.grid(True, linestyle=':')

plt.tight_layout()
plt.show()

In [20]:
# Section 20: Governance & Output Contract Validation
governance_notice = {
    "model": "MODEL 9 — MATERNAL & CLINICAL CONTEXT ENGINE",
    "is_diagnostic": False,
    "intended_use": "Longitudinal feature engineering for Pregnancy Digital Twin and multi-modal risk models",
    "disclaimer": "Model 9 provides structured non-diagnostic maternal context and does not make autonomous clinical diagnoses."
}
print("Clinical Governance Verified:", json.dumps(governance_notice, indent=2))